In [2]:
import time # Para pausar entre páginas
import re # Para limpiar el precio
from dataclasses import dataclass, field # Comprimir las clases
from typing import Optional # Puede devolver valor o None

import pandas as pd #Para trabajar con datos tabulares
from IPython.display import display # Para mostrar un formato tabla
from selenium import webdriver # El acceso al Chrome
from selenium.webdriver.chrome.service import Service # El puende de Selenium a Chrome
from selenium.webdriver.chrome.options import Options # Para configurar el Chrome
from selenium.webdriver.common.by import By # El tipo de selector para encontrar elementos
from selenium.webdriver.support.ui import WebDriverWait #Espera al elemento que aparezca en la pagina
from selenium.webdriver.support import expected_conditions as EC # Define las condiciones (esté presente, sea clickeable, sea visible, etc)
from webdriver_manager.chrome import ChromeDriverManager # descarga la versión correcta de chromedriver

@dataclass # Es una forma de escribir clases que solo sirven para guardar datos
class Config:
    url_inicial: str        = "https://books.toscrape.com/" #Primera página que visitará el scraper
    paginas_objetivo: int   = 5 # Cuántas páginas se recorreran
    timeout_elemento: int   = 10 # Segundos máx deespera de carga
    timeout_navegacion: int = 5 # Segundos máx de espera del botón next
    pausa_entre_paginas: float = 2.0 # Pausa entre pág
    archivo_salida: str     = "libros_scrapeados" # Nombre para los achivos de salida

    sel_contenedor:    str = "article.product_pod" # Selector CSS del contenedor de cada libro en la página
    sel_titulo:        str = "h3 a" # Selector del elemento que contiene el título y la URL del libro
    sel_precio:        str = "p.price_color" # Selector del precio
    sel_rating:        str = "p.star-rating" # Selector de rating
    sel_disponibilidad:str = "p.availability" # Selector de la disponibilidad
    xpath_siguiente:   str = "//li[@class='next']/a" # XPATH del botón de siguiente página

    rating_map: dict = field(default_factory=lambda: { # Diccionario que convierte la palabra del rating a su número.
        "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
    })

cfg = Config(paginas_objetivo=5) # Creamos una instancia de Config con los valores por defecto.
print("Configuración lista")
print(f"   URL        : {cfg.url_inicial}")
print(f"   Páginas    : {cfg.paginas_objetivo}")
print(f"   Pausa/pág  : {cfg.pausa_entre_paginas}s")

Configuración lista
   URL        : https://books.toscrape.com/
   Páginas    : 5
   Pausa/pág  : 2.0s


In [6]:
def crear_driver(cfg: Config) -> webdriver.Chrome: 
    opciones = Options() # Se crea el objeto de opc de chrome
    opciones.binary_location = "/usr/bin/google-chrome" # Ruta ejecutable de Chrome en el Docker
    opciones.add_argument("--headless") # Se ejecuta sin interfaz visual
    opciones.add_argument("--no-sandbox") #desactiva el sandbox de seguridad de Chrome
    opciones.add_argument("--disable-dev-shm-usage") # evita que Chrome use /dev/shm (memoria compartida).En Docker ese espacio es muy pequeño y causa crashes sin esta opción
    opciones.add_argument("--window-size=1920,1080") # devine el tamaño de la ventana (para no ocultar elementos)
    opciones.add_argument( # cadena que identifica el navegador ante el servidor
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome( 
        service=Service(ChromeDriverManager().install()), # Descarga y gestiona automáticamente el chromedriver compatible con la versión de Chrome
        options=opciones, # Aplica toda la configuración anterior
    )
    return driver

driver = crear_driver(cfg) # Llamamos a la función y guardamos el driver en una variable global para poder usarlo en las celdas siguientes
print("Driver inicializado")

Driver inicializado


In [7]:
def extraer_libro(libro, pagina: int, cfg: Config) -> Optional[dict]:
    """Extrae los campos de un article.product_pod. Devuelve None si falla."""
    try:
        titulo_el = libro.find_element(By.CSS_SELECTOR, cfg.sel_titulo) # se busca <h3><a> dentro del contenedor del libro
        titulo    = titulo_el.get_attribute("title").strip() # obtiene el atributo HTML 'title' del enlace, que contiene el nombre completo
        url_libro = titulo_el.get_attribute("href") # obtiene la URL a la página de detalle del libro

        precio_texto = libro.find_element(By.CSS_SELECTOR, cfg.sel_precio).text # Texto delprecio
        precio = float(re.sub(r"[^\d.]", "", precio_texto))   # limpieza robusta

        rating_class = libro.find_element( # El rating no esta en texto sino en la clase CSS del elemento
            By.CSS_SELECTOR, cfg.sel_rating
        ).get_attribute("class")
        rating = next( # Busca en el diccionario
            (v for k, v in cfg.rating_map.items() if k in rating_class), 0
        )

        try:
            disponibilidad = libro.find_element( # Busca la disponibilidad
                By.CSS_SELECTOR, cfg.sel_disponibilidad
            ).text.strip()
        except Exception:
            disponibilidad = "Desconocida"

        return {
            "pagina":        pagina,
            "titulo":        titulo,
            "precio_gbp":    precio,
            "rating":        rating,
            "disponibilidad":disponibilidad,
            "url":           url_libro,
        }
    except Exception as e:
        print(f"  Error extrayendo libro en pág {pagina}: {e}")
        return None

print("Función extraer_libro definida")

Función extraer_libro definida


In [8]:
datos_totales = []

try:
    print(f"Navegando a {cfg.url_inicial}\n")
    driver.get(cfg.url_inicial) # Se abre la URL inicial en el navegador controlado

    for pagina in range(1, cfg.paginas_objetivo + 1): # 'pagina' va de 1 hasta paginas_objetivo
        print(f"{'─'*50}")
        print(f"  Página {pagina} / {cfg.paginas_objetivo}")
        print(f"{'─'*50}")

        # Esperar elementos
        try:
            WebDriverWait(driver, cfg.timeout_elemento).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, cfg.sel_contenedor)
                )
            )
        except Exception:
            print(f"  Timeout en página {pagina}. Deteniendo.")
            break

        libros = driver.find_elements(By.CSS_SELECTOR, cfg.sel_contenedor) # Devuelve una lista de todos los elementos que coinciden con el selector
        print(f"  Libros encontrados : {len(libros)}")

        errores = 0
        for libro in libros:
            resultado = extraer_libro(libro, pagina, cfg) # Llama a la función extraer
            if resultado:
                datos_totales.append(resultado) # Añade a la lista
            else:
                errores += 1

        print(f"  Extraídos OK       : {len(libros) - errores}")
        print(f"  Errores            : {errores}")
        print(f"  Total acumulado    : {len(datos_totales)}\n")

        if pagina >= cfg.paginas_objetivo: # Se sale del bucle si se llega a la última página
            print("  Límite de páginas alcanzado.")
            break

        # Navegar a la siguiente página
        try:
            boton = WebDriverWait(driver, cfg.timeout_navegacion).until( # Se espera que el botón sea clickleable
                EC.element_to_be_clickable((By.XPATH, cfg.xpath_siguiente))
            )
            boton.click()
            time.sleep(cfg.pausa_entre_paginas)
        except Exception:
            print("  ℹ Botón 'Next' no encontrado. Última página alcanzada.")
            break

except Exception as e:
    print(f"Error inesperado: {e}")
    import traceback; traceback.print_exc()

finally:
    driver.quit()
    print("\nNavegador cerrado de forma segura.")

Navegando a https://books.toscrape.com/

──────────────────────────────────────────────────
  Página 1 / 5
──────────────────────────────────────────────────
  Libros encontrados : 20
  Extraídos OK       : 20
  Errores            : 0
  Total acumulado    : 20

──────────────────────────────────────────────────
  Página 2 / 5
──────────────────────────────────────────────────
  Libros encontrados : 20
  Extraídos OK       : 20
  Errores            : 0
  Total acumulado    : 40

──────────────────────────────────────────────────
  Página 3 / 5
──────────────────────────────────────────────────
  Libros encontrados : 20
  Extraídos OK       : 20
  Errores            : 0
  Total acumulado    : 60

──────────────────────────────────────────────────
  Página 4 / 5
──────────────────────────────────────────────────
  Libros encontrados : 20
  Extraídos OK       : 20
  Errores            : 0
  Total acumulado    : 80

──────────────────────────────────────────────────
  Página 5 / 5
─────────

In [9]:
if datos_totales:
    df = pd.DataFrame(datos_totales) # Convierte la lista en una tabla

    csv_path  = f"{cfg.archivo_salida}.csv" # Rutas
    xlsx_path = f"{cfg.archivo_salida}.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig") # Se guarda en formato CSV
    df.to_excel(xlsx_path, index=False) # Se guarda en formato excel

    print(f"Guardado: {csv_path}")
    print(f"Guardado: {xlsx_path}")
    print(f"   Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas\n")

    display(df.head(10))
else:
    print("No se capturaron datos.")

Guardado: libros_scrapeados.csv
Guardado: libros_scrapeados.xlsx
   Dimensiones: 100 filas × 6 columnas



,pagina,titulo,precio_gbp,rating,disponibilidad,url
0,1,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,1,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
3,1,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,1,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,1,The Requiem Red,22.65,1,In stock,https://books.toscrape.com/catalogue/the-requi...
6,1,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,1,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,https://books.toscrape.com/catalogue/the-comin...
8,1,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,1,The Black Maria,52.15,1,In stock,https://books.toscrape.com/catalogue/the-black...


In [10]:
print("Registros por página:")
display(df['pagina'].value_counts().sort_index().rename('libros')) # value (cuenta cuantas veces aparece cada valor en la 'pagina'), index (ordena por número de página), rename (cambia el nombre de la serie para mejor visualización)

print("\nEstadísticas de precio (£):")
display(df['precio_gbp'].describe().round(2)) # calcula: count, mean, std, min, 25%, 50%, 75%, max

Registros por página:


pagina
1    20
2    20
3    20
4    20
5    20
Name: libros, dtype: int64


Estadísticas de precio (£):


count    100.00
mean      34.56
std       14.64
min       10.16
25%       19.90
50%       34.78
75%       47.97
max       58.11
Name: precio_gbp, dtype: float64

In [11]:
print("Distribución de ratings:")
rating_df = (
    df['rating']
    .value_counts() # cuenta libros por cada valor de rating
    .sort_index() # ordena de 1 a 5 estrellas
    .rename_axis('estrellas') # renombra el índice
    .reset_index(name='cantidad') # convierte el índice en columna
)
rating_df['visual'] = rating_df['estrellas'].apply(
    lambda n: '★' * n + '☆' * (5 - n) # Columna visual para las estrellas
)
display(rating_df)

print("\nDisponibilidad:")
display(df['disponibilidad'].value_counts())

Distribución de ratings:


,estrellas,cantidad,visual
0,1,22,★☆☆☆☆
1,2,19,★★☆☆☆
2,3,22,★★★☆☆
3,4,18,★★★★☆
4,5,19,★★★★★



Disponibilidad:


disponibilidad
In stock    100
Name: count, dtype: int64

In [12]:
print("Top 5 libros más caros:")
display(df.nlargest(5, 'precio_gbp')[['titulo', 'precio_gbp', 'rating']]) # Selecciona y muestra las 5 filas de precio mas alto

print("\nTop 5 libros más baratos:")
display(df.nsmallest(5, 'precio_gbp')[['titulo', 'precio_gbp', 'rating']]) #Selecciona y muestra las 5 filas de precio mas bajo

Top 5 libros más caros:


,titulo,precio_gbp,rating
68,The Death of Humanity: and the Case for Life,58.11,4
40,Slow States of Collapse: Poems,57.31,3
15,Our Band Could Be Your Life: Scenes from the A...,57.25,3
58,The Past Never Ends,56.50,4
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1



Top 5 libros más baratos:


,titulo,precio_gbp,rating
84,Patience,10.16,3
20,In Her Wake,12.84,1
81,Princess Between Worlds (Wide-Awake Princess #5),13.34,5
80,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5
10,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99,2
